### **Projeto de Análise de Vendas e Lucratividade - Superstore**
**Objetivo:** Limpar, tratar e preparar os dados brutos de vendas para análise exploratória. O foco é garantir a qualidade da informação para identificar padrões de lucro, prejuízo e impacto dos descontos aplicados.

## 1. Importação e Carregamento de Dados
Nesta etapa inicial, importamos a biblioteca Pandas e carregamos o ficheiro original (bruto) para o nosso ambiente de trabalho.

In [1]:
import pandas as pd

#Importando o Dataset
df = pd.read_csv("../data/raw/Sample - Superstore.csv", encoding='latin1')

## 2. Inspeção Inicial dos Dados
Vamos visualizar as primeiras linhas e entender a estrutura do conjunto de dados, analisando os tipos de variáveis e o volume total de registos.

In [2]:
#Breve visualização do Dataframe
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 3. Verificação da Qualidade dos Dados (Data Quality)
Para garantir cálculos precisos e visualizações fiáveis, é fundamental verificar a existência de valores nulos (dados faltantes) ou registos duplicados que possam enviesar ou distorcer a nossa análise.

In [3]:
#Verificando se existe algum valor nulo nas colunas
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [4]:
#Verificando se há tuplas duplicadas
df.duplicated(subset=df.columns.drop('Row ID')).sum()

np.int64(1)

In [5]:
#Atualizando o Dataframe caso houvesse linhas duplicadas
df = df.drop_duplicates(subset=df.columns.drop('Row ID'))
df.duplicated(subset=df.columns.drop('Row ID')).sum()

np.int64(0)

In [6]:
#Atualizando index depois da remoção de linhas duplicadas
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9993 non-null   int64  
 1   Order ID       9993 non-null   object 
 2   Order Date     9993 non-null   object 
 3   Ship Date      9993 non-null   object 
 4   Ship Mode      9993 non-null   object 
 5   Customer ID    9993 non-null   object 
 6   Customer Name  9993 non-null   object 
 7   Segment        9993 non-null   object 
 8   Country        9993 non-null   object 
 9   City           9993 non-null   object 
 10  State          9993 non-null   object 
 11  Postal Code    9993 non-null   int64  
 12  Region         9993 non-null   object 
 13  Product ID     9993 non-null   object 
 14  Category       9993 non-null   object 
 15  Sub-Category   9993 non-null   object 
 16  Product Name   9993 non-null   object 
 17  Sales          9993 non-null   float64
 18  Quantity

> **Resumo da Qualidade:** Os dados foram verificados. As linhas duplicadas foram removidas com sucesso e confirmámos a ausência de valores nulos, garantindo a integridade do conjunto de dados para as próximas etapas.

## 4. Padronização e Tradução (Localização)
Para facilitar a compreensão do negócio e melhorar o *storytelling* dos nossos gráficos, todas as colunas principais e os seus respetivos valores categóricos serão traduzidos para português.

In [7]:
#Traduzindo para português as colunas e valores mais relevantes
traducao_colunas = {
    'Row ID': 'ID',
    'Order ID': 'ID_Pedido',
    'Order Date': 'Data_Pedido',
    'Ship Date': 'Data_Envio',
    'Ship Mode': 'Modo_Envio',
    'Customer ID': 'ID_Cliente',
    'Customer Name': 'Nome_Cliente',
    'Segment': 'Segmento',
    'Country': 'Pais',
    'City': 'Cidade',
    'State': 'Estado',
    'Postal Code': 'Codigo_Postal',
    'Region': 'Regiao',
    'Product ID': 'ID_Produto',
    'Category': 'Categoria',
    'Sub-Category': 'Sub_Categoria',
    'Product Name': 'Nome_Produto',
    'Sales': 'Vendas',
    'Quantity': 'Quantidade',
    'Discount': 'Desconto',
    'Profit': 'Lucro'
}

traducao_valores = {
    'Modo_Envio': {
        'Standard Class': 'Classe Padrao',
        'Second Class': 'Segunda Classe',
        'First Class': 'Primeira Classe',
        'Same Day': 'Mesmo Dia'
    },
    'Segmento': {
        'Consumer': 'Consumidor',
        'Corporate': 'Corporativo',
        'Home Office': 'Home Office'
    },
    'Pais': {
      'United States': 'Estados Unidos'  
    },
    'Regiao': {
        'South': 'Sul',
        'West': 'Oeste',
        'Central': 'Central',
        'East': 'Leste'
    },
    'Categoria': {
        'Furniture': 'Mobilia',
        'Office Supplies': 'Materiais de Escritorio',
        'Technology': 'Tecnologia'
    },
    'Sub_Categoria': {
        'Tables': 'Mesas',
        'Chairs': 'Cadeiras',
        'Phones': 'Telefones',
        'Storage': 'Armazenamento',
        'Binders': 'Pastas',
        'Machines': 'Máquinas',
        'Accessories': 'Acessorios',
        'Bookcases': 'Estantes',
        'Appliances': 'Eletrodomesticos',
        'Supplies': 'Suprimentos',
        'Envelopes': 'Envelopes',
        'Art': 'Arte',
        'Labels': 'Etiquetas',
        'Paper': 'Papel',
        'Fasteners': 'Prendedores',
        'Furnishings': 'Decoracao',
        'Copiers': 'Copiadoras'
    }
}

#Aplicando a tradução
df = df.rename(columns=traducao_colunas)
df.replace(traducao_valores, inplace=True)

#Visualizando as alterações
df.head()

,ID,ID_Pedido,Data_Pedido,Data_Envio,Modo_Envio,ID_Cliente,Nome_Cliente,Segmento,Pais,Cidade,...,Codigo_Postal,Regiao,ID_Produto,Categoria,Sub_Categoria,Nome_Produto,Vendas,Quantidade,Desconto,Lucro
0,1,CA-2016-152156,11/8/2016,11/11/2016,Segunda Classe,CG-12520,Claire Gute,Consumidor,Estados Unidos,Henderson,...,42420,Sul,FUR-BO-10001798,Mobilia,Estantes,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Segunda Classe,CG-12520,Claire Gute,Consumidor,Estados Unidos,Henderson,...,42420,Sul,FUR-CH-10000454,Mobilia,Cadeiras,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Segunda Classe,DV-13045,Darrin Van Huff,Corporativo,Estados Unidos,Los Angeles,...,90036,Oeste,OFF-LA-10000240,Materiais de Escritorio,Etiquetas,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Classe Padrao,SO-20335,Sean O'Donnell,Consumidor,Estados Unidos,Fort Lauderdale,...,33311,Sul,FUR-TA-10000577,Mobilia,Mesas,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Classe Padrao,SO-20335,Sean O'Donnell,Consumidor,Estados Unidos,Fort Lauderdale,...,33311,Sul,OFF-ST-10000760,Materiais de Escritorio,Armazenamento,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 4.1. Normalização do tipo Data
Para que tenhamos uma análise mais consistente, temos que verificar se as colunas de Data estão no tipo de dado adequado. Isso nos permite a manipulação correta dos dados e que as funções funcionem do modo desejado.

In [8]:
#Exibindo o tipo de cada coluna do Dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID             9993 non-null   int64  
 1   ID_Pedido      9993 non-null   object 
 2   Data_Pedido    9993 non-null   object 
 3   Data_Envio     9993 non-null   object 
 4   Modo_Envio     9993 non-null   object 
 5   ID_Cliente     9993 non-null   object 
 6   Nome_Cliente   9993 non-null   object 
 7   Segmento       9993 non-null   object 
 8   Pais           9993 non-null   object 
 9   Cidade         9993 non-null   object 
 10  Estado         9993 non-null   object 
 11  Codigo_Postal  9993 non-null   int64  
 12  Regiao         9993 non-null   object 
 13  ID_Produto     9993 non-null   object 
 14  Categoria      9993 non-null   object 
 15  Sub_Categoria  9993 non-null   object 
 16  Nome_Produto   9993 non-null   object 
 17  Vendas         9993 non-null   float64
 18  Quantida

In [9]:
#Normalizando o tipo de dado das colunas de data
df['Data_Pedido'] = pd.to_datetime(df['Data_Pedido'])
df['Data_Envio'] = pd.to_datetime(df['Data_Envio'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ID             9993 non-null   int64         
 1   ID_Pedido      9993 non-null   object        
 2   Data_Pedido    9993 non-null   datetime64[ns]
 3   Data_Envio     9993 non-null   datetime64[ns]
 4   Modo_Envio     9993 non-null   object        
 5   ID_Cliente     9993 non-null   object        
 6   Nome_Cliente   9993 non-null   object        
 7   Segmento       9993 non-null   object        
 8   Pais           9993 non-null   object        
 9   Cidade         9993 non-null   object        
 10  Estado         9993 non-null   object        
 11  Codigo_Postal  9993 non-null   int64         
 12  Regiao         9993 non-null   object        
 13  ID_Produto     9993 non-null   object        
 14  Categoria      9993 non-null   object        
 15  Sub_Categoria  9993 n

## 4.2. Arrumando valores com espaços extras
Por vezes, os dados brutos contém espaços extras desnecessários antes ou depois do texto, e isso pode ocasionar distorções e inconsistências na análise. Portanto, é importante que os espaços sejam removidos e os valores sejam arrumados (semelhante a função =ARRUMAR() do Excel).

In [10]:
#Removendo possíveis espaços desnecessários em cada linha de tipo texto
for coluna in df.columns:
    if df[coluna].dtype == "object":
        df[coluna] = df[coluna].str.strip()

## 5. Engenharia de Funcionalidades (Feature Engineering)
Com os dados limpos, criamos novas métricas essenciais para o negócio. Adicionámos a coluna `Margem de Lucro %` para entender a eficiência de cada venda, não olhando apenas para o volume absoluto.

In [11]:
#Adicionando nova coluna de Margem de Lucro como tipo float
df['Margem de Lucro %'] = (df['Lucro'] / df['Vendas'])*100
df['Margem de Lucro %'] = df['Margem de Lucro %'].astype(float).round(2)

In [12]:
#Exibição das 10 primeiras linhas para visualizar o resultado
df.head(10)

,ID,ID_Pedido,Data_Pedido,Data_Envio,Modo_Envio,ID_Cliente,Nome_Cliente,Segmento,Pais,Cidade,...,Regiao,ID_Produto,Categoria,Sub_Categoria,Nome_Produto,Vendas,Quantidade,Desconto,Lucro,Margem de Lucro %
0,1,CA-2016-152156,2016-11-08,2016-11-11,Segunda Classe,CG-12520,Claire Gute,Consumidor,Estados Unidos,Henderson,...,Sul,FUR-BO-10001798,Mobilia,Estantes,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,16.00
1,2,CA-2016-152156,2016-11-08,2016-11-11,Segunda Classe,CG-12520,Claire Gute,Consumidor,Estados Unidos,Henderson,...,Sul,FUR-CH-10000454,Mobilia,Cadeiras,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,30.00
2,3,CA-2016-138688,2016-06-12,2016-06-16,Segunda Classe,DV-13045,Darrin Van Huff,Corporativo,Estados Unidos,Los Angeles,...,Oeste,OFF-LA-10000240,Materiais de Escritorio,Etiquetas,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,47.00
3,4,US-2015-108966,2015-10-11,2015-10-18,Classe Padrao,SO-20335,Sean O'Donnell,Consumidor,Estados Unidos,Fort Lauderdale,...,Sul,FUR-TA-10000577,Mobilia,Mesas,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-40.00
4,5,US-2015-108966,2015-10-11,2015-10-18,Classe Padrao,SO-20335,Sean O'Donnell,Consumidor,Estados Unidos,Fort Lauderdale,...,Sul,OFF-ST-10000760,Materiais de Escritorio,Armazenamento,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,11.25
5,6,CA-2014-115812,2014-06-09,2014-06-14,Classe Padrao,BH-11710,Brosina Hoffman,Consumidor,Estados Unidos,Los Angeles,...,Oeste,FUR-FU-10001487,Mobilia,Decoracao,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694,29.00
6,7,CA-2014-115812,2014-06-09,2014-06-14,Classe Padrao,BH-11710,Brosina Hoffman,Consumidor,Estados Unidos,Los Angeles,...,Oeste,OFF-AR-10002833,Materiais de Escritorio,Arte,Newell 322,7.2800,4,0.00,1.9656,27.00
7,8,CA-2014-115812,2014-06-09,2014-06-14,Classe Padrao,BH-11710,Brosina Hoffman,Consumidor,Estados Unidos,Los Angeles,...,Oeste,TEC-PH-10002275,Tecnologia,Telefones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152,10.00
8,9,CA-2014-115812,2014-06-09,2014-06-14,Classe Padrao,BH-11710,Brosina Hoffman,Consumidor,Estados Unidos,Los Angeles,...,Oeste,OFF-BI-10003910,Materiais de Escritorio,Pastas,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825,31.25
9,10,CA-2014-115812,2014-06-09,2014-06-14,Classe Padrao,BH-11710,Brosina Hoffman,Consumidor,Estados Unidos,Los Angeles,...,Oeste,OFF-AP-10002892,Materiais de Escritorio,Eletrodomesticos,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5,0.00,34.4700,30.00


## 6. Exportação do Conjunto de Dados Tratado
Após todas as transformações, o conjunto de dados está consolidado. Exportamos o resultado final para um novo arquivo CSV, que servirá de base limpa e segura para a criação dos gráficos e indicadores do projeto.

In [13]:
#Exportando Dataset "limpo" em csv
df.to_csv("../data/processed/SampleSuperstore_Tratado.csv", index=False)